<a href="https://www.kaggle.com/code/leonardoterra/shopify-revenue-forecast?scriptVersionId=309647445" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Loading Libraries & Checking Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Libraries for regression models, including OLS
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from scipy import stats

warnings.filterwarnings('ignore')

In [ ]:
dataframe = pd.read_csv('/kaggle/input/datasets/leonardoterra/monthly-revenue-and-spend/shopify_monthly_revenue.csv')
dataframe.info()

In [ ]:
# For SARIMA model

df_log = dataframe
df_log["revenue"] = pd.to_numeric(df_log["revenue"])
df_log["revenue_log"] = np.log(df_log["revenue"])

In [ ]:
# Train & Test split
y = dataframe['revenue']
X = dataframe.drop(['date','revenue'], axis=1)

# Add a constant (intercept) to the model
X = sm.add_constant(X)

# Fit the OLS model
model = sm.OLS(y, X)
results = model.fit()

# Print results
print(results.summary())

# Preparing Data for ML Models

In [ ]:
# Librearies and function to get metrics

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    max_error
)

def regression_metrics(y_true, y_pred, round_digits=4):
    print("Evaluation Metrics:\n")

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    max_err = max_error(y_true, y_pred)

    print(f"R² Score: {r2:.{round_digits}f}")
    print(f"MAE: {mae:.{round_digits}f}")
    print(f"MAPE(%): {mape * 100:.{round_digits}f}%")
    print(f"Max Error: {max_err:.{round_digits}f}")

In [ ]:
# @title

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
y = dataframe['revenue']
X = dataframe.drop(['date','revenue','revenue_log'], axis=1)

# Time-respectingsplit
cutoff = 5

X_train, X_test = X.iloc[:-cutoff], X.iloc[-cutoff:]
y_train, y_test = y.iloc[:-cutoff], y.iloc[-cutoff:]

# Initialize the StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Loading Models & Metrics

In [ ]:
# @title
'''
# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Print metrics
regression_metrics(y_test, y_pred, round_digits=2)
'''

In [ ]:
# @title
'''
pd.options.display.float_format = '{:.2f}'.format
print(pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}))
'''

**XGB Model**

In [ ]:
from xgboost import XGBRegressor

# Define features and target
y = dataframe['revenue']
X = dataframe.drop(['date','revenue','revenue_log'], axis=1)

# 3. Train XGBoost
model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
model_xgb.fit(X_train, y_train)

# 4. Predict and evaluate
y_pred = model_xgb.predict(X_test)
regression_metrics(y_test, y_pred, round_digits=2)

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
print(pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}))

**SARIMA**

In [ ]:
# @title
!pip install pmdarima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
'''

# @title
# Decompose the series
result = seasonal_decompose(df_log['revenue_log'], model='additive', period=12)

# Add components as columns
df_log['trend'] = result.trend
df_log['seasonal'] = result.seasonal
df_log['residual'] = result.resid

# See the result
result.plot()
plt.gcf().set_size_inches(12, 8)
plt.show()

'''

In [ ]:
'''
# @title
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Autocorrelation check
fig, ax = plt.subplots(2, 1, figsize=(12,8))
plot_acf(df_log['revenue_log'], lags=12, ax=ax[0])
plot_pacf(df_log['revenue_log'], lags=12, ax=ax[1])
plt.show()
'''

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df_log['revenue_log'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])

# p-value < 0.05 → The series **is stationary** (rejects non-stationary).
# p-value ≥ 0.05 → The series **is non-stationary** (you need to fix it).

In [ ]:
'''

# @title
#Auto Arima
from pmdarima import auto_arima

auto_model = auto_arima(
    df_log['revenue_log'],
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,
    seasonal=True, m=12, # ← monthly seasonality within weekly data
    trace=True,
    information_criterion='aic'
)
print(auto_model.summary())
'''

In [ ]:
# @title
from statsmodels.tsa.statespace.sarimax import SARIMAX

# --- Train/test split ---
train = df_log.iloc[:-1]
test = df_log.iloc[-1:]

# SARIMA
sarima_model = SARIMAX(
    train['revenue_log'],
    order=(0, 1, 1),              # (p, d, q)
    seasonal_order=(1, 0, 0, 12), # (P, D, Q, m)
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

# Forecast
sarima_forecast = sarima_fit.forecast(steps=1)

# Compare forecasts vs actuals
results = pd.DataFrame({
    'actual': test['revenue_log'],
    'sarima': sarima_forecast.values
})

mae_sarimax = (results['actual'] - results['sarima']).abs().mean()
print(f"MAE — SARIMAX: {mae_sarimax:.2f}")

# Diagnostics
#fig, axes = plt.subplots(2, 2, figsize=(20, 8))
#sarima_fit.plot_diagnostics(fig=fig)
#plt.suptitle('SARIMA Residual Diagnostics')
#plt.tight_layout()
#plt.show()

In [ ]:
'''
# @title
results['actual'] = np.exp(results['actual'])
results['sarima'] = np.exp(results['sarima'])
results
'''

In [ ]:
# Future forecast - SARIMA MODEL

n_months = 1
history = list(df_log['revenue_log'])
future_predictions = []

for i in range(n_months):
    sarima_model = SARIMAX(
      history,
      order=(0, 1, 1),              # (p, d, q)
      seasonal_order=(1, 0, 0, 12), # (P, D, Q, m)
      enforce_stationarity=False,
      enforce_invertibility=False
)
    sarima_fit = sarima_model.fit(disp=False)

    # Forecast 1 step
    sarima_pred = sarima_fit.forecast(steps=1)

# Results table
results_future = pd.DataFrame({
    'Next Week': range(1, n_months + 1),
    'predicted_revenue': np.exp(sarima_pred).round(2)
})

print(results_future.to_string(index=False))

# Forecasting

In [ ]:
# Future prediction -Regression Models

# Variables

period=28
seasonality=4
year=2026
quarter=2
spend=295000
promo=0

next_X = pd.DataFrame([[period, seasonality, year, quarter, spend, promo]], columns=X.columns)
next_X_scaled = scaler.transform(next_X)

# regression_pred = model.predict(next_X_scaled) # Linear regression
xgb_pred = model_xgb.predict(next_X) # XGBoost
sarimax_pred = np.exp(sarima_pred)[0].round(2) # Sarima

In [ ]:
y_pred_ensemble = (xgb_pred[0] + sarimax_pred) / 2

# print(f"Linear Regression: {regression_pred[0]:,.2f}")
print(f"XGBoost: {xgb_pred[0]:,.2f}")
print(f"SARIMA: {sarimax_pred:,.2f}")
print(f"Ensemble: {y_pred_ensemble:,.2f}")